In [ ]:
# @title 1. Install Dependencies
!pip install -q transformers accelerate torch sentence-transformers scikit-learn tqdm pandas numpy

In [ ]:
# @title 2. Imports and Configuration
import re
import json
import torch
import numpy as np
import pandas as pd
from datetime import datetime
from typing import List, Dict, Tuple, Optional
from collections import defaultdict, Counter
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# Configuration
MODEL_NAME = "athena129/CyberSecQwen-4B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LOGS_FOR_ANALYSIS = 10000
MAX_GROUPS_TO_ANALYZE = 5
EMBEDDING_BATCH_SIZE = 32

print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# @title 3. Core Analyzer Class (FIXED - Level Detection)
class LogAnalyzer:
    """
    AI-powered log analyzer using CyberSecQwen-4B
    """

    # Known security event types
    SECURITY_EVENT_TYPES = [
        'DDOS', 'XSS', 'SQL_INJECTION', 'SCALE_INJECTION', 'TG',
        'MALWARE', 'PHISHING', 'RANSOMWARE', 'BRUTE_FORCE', 'PORT_SCAN',
        'BACKDOOR', 'CMD_INJECTION', 'PATH_TRAVERSAL', 'CSRF', 'SSRF'
    ]

    def __init__(self):
        print("Loading models...")

        # Load CyberSecQwen-4B
        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        self.model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
            device_map="auto" if DEVICE == "cuda" else None,
            low_cpu_mem_usage=True
        )

        # Load embedding model for clustering
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')

        # Regex patterns for structured field extraction
        self.field_patterns = {
            'level': re.compile(r'\b(?:level|severity|log_level)\s*[=:]\s*(\w+)', re.IGNORECASE),
            'type': re.compile(r'\b(?:type|event_type|attack_type|threat_type)\s*[=:]\s*(\w+)', re.IGNORECASE),
            'source_ip': re.compile(r'\b(?:source_ip|src_ip|from_ip|client_ip)\s*[=:]\s*(\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})', re.IGNORECASE),
            'dest_ip': re.compile(r'\b(?:dest_ip|dst_ip|to_ip|server_ip|target_ip)\s*[=:]\s*(\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})', re.IGNORECASE),
            'status': re.compile(r'\b(?:status|status_code|http_status)\s*[=:]\s*(\w+)', re.IGNORECASE),
            'request_id': re.compile(r'\b(?:request_id|req_id|trace_id|transaction_id)\s*[=:]\s*([\w-]+)', re.IGNORECASE),
            'action': re.compile(r'\b(?:action|response_action|firewall_action)\s*[=:]\s*(\w+)', re.IGNORECASE),
        }

        print("Models loaded successfully!")

    def extract_structured_fields(self, line: str) -> Dict:
        """Extract structured fields from log line using regex"""
        fields = {}

        for field_name, pattern in self.field_patterns.items():
            match = pattern.search(line)
            if match:
                fields[field_name] = match.group(1).strip()

        return fields

    def extract_level(self, line: str, fields: Optional[Dict] = None) -> str:
        """Extract severity level from log line"""
        if fields is None:
            fields = self.extract_structured_fields(line)

        # Priority 1: explicit level field (level=ERROR)
        if 'level' in fields:
            level_raw = fields['level'].upper()
            if level_raw in ['CRITICAL', 'FATAL', 'SEVERE', 'EMERGENCY', 'ALERT']:
                return 'CRITICAL'
            elif level_raw in ['ERROR', 'ERR', 'FAIL', 'FAILURE']:
                return 'ERROR'
            elif level_raw in ['WARNING', 'WARN']:
                return 'WARNING'
            elif level_raw in ['INFO', 'DEBUG', 'TRACE', 'NOTICE']:
                return 'INFO'

        # Priority 2: Look for standalone level words in the line
        # Pattern: word boundary + LEVEL + word boundary
        line_upper = line.upper()

        if re.search(r'\bCRITICAL\b|\bFATAL\b|\bSEVERE\b|\bEMERGENCY\b', line_upper):
            return 'CRITICAL'
        elif re.search(r'\bERROR\b|\bERR\b|\bFAILED\b|\bFAILURE\b|\bEXCEPTION\b', line_upper):
            return 'ERROR'
        elif re.search(r'\bWARNING\b|\bWARN\b', line_upper):
            return 'WARNING'
        elif re.search(r'\bINFO\b', line_upper):
            return 'INFO'
        elif re.search(r'\bDEBUG\b|\bTRACE\b', line_upper):
            return 'INFO'  # DEBUG/TRACE treated as INFO (not important)

        # Priority 3: Fallback - check for error keywords in message
        line_lower = line.lower()
        if any(word in line_lower for word in ['error', 'fail', 'exception']):
            return 'ERROR'
        elif any(word in line_lower for word in ['warn', 'warning']):
            return 'WARNING'
        else:
            return 'INFO'

    def extract_security_event_type(self, line: str, fields: Optional[Dict] = None) -> Optional[str]:
        """Extract security event type from log line"""
        if fields is None:
            fields = self.extract_structured_fields(line)

        if 'type' in fields:
            type_raw = fields['type'].upper()
            for event_type in self.SECURITY_EVENT_TYPES:
                if event_type in type_raw or type_raw in event_type:
                    return event_type

        line_upper = line.upper()
        for event_type in self.SECURITY_EVENT_TYPES:
            if event_type in line_upper:
                return event_type

        return None

    def extract_timestamp(self, line: str) -> str:
        """Extract timestamp from log line"""
        patterns = [
            r'\d{4}-\d{2}-\d{2}[\sT]\d{2}:\d{2}:\d{2}',
            r'\d{2}/\w+/\d{4}:\d{2}:\d{2}:\d{2}',
            r'\w+\s+\d+\s+\d+:\d+:\d+'
        ]

        for pattern in patterns:
            match = re.search(pattern, line)
            if match:
                return match.group(0)

        return 'unknown'

    def parse_logs(self, content: str) -> List[Dict]:
        """Parse logs and extract structured information"""
        lines = content.strip().split('\n')

        important_logs = []
        total_lines = len(lines)

        print(f"Parsing {total_lines} lines...")

        for line in lines:
            line = line.strip()
            if not line:
                continue

            fields = self.extract_structured_fields(line)
            level = self.extract_level(line, fields)
            event_type = self.extract_security_event_type(line, fields)

            # Only keep ERROR, CRITICAL, WARNING
            if level in ['ERROR', 'CRITICAL', 'WARNING']:
                important_logs.append({
                    'level': level,
                    'message': line,
                    'timestamp': self.extract_timestamp(line),
                    'fields': fields,
                    'event_type': event_type
                })

        return important_logs

    def analyze_ngrams(self, logs: List[Dict], top_k: int = 20, sample_size: int = 500) -> Dict:
        """Optimized n-gram analysis"""
        if len(logs) > sample_size:
            step = len(logs) // sample_size
            sampled_logs = logs[::step][:sample_size]
        else:
            sampled_logs = logs

        pattern_counter = Counter()

        for log in sampled_logs:
            fields = log.get('fields', {})
            event_type = log.get('event_type')

            if event_type:
                pattern_counter[f"type={event_type}"] += 1

            key_fields = ['level', 'source_ip', 'dest_ip', 'status', 'action']
            field_parts = []
            for field_name in key_fields:
                if field_name in fields:
                    field_parts.append(f"{field_name}={fields[field_name]}")

            if field_parts:
                pattern_counter[' '.join(field_parts)] += 1

        meaningful_patterns = {p: c for p, c in pattern_counter.items() if c >= 2}
        top_patterns = sorted(meaningful_patterns.items(), key=lambda x: x[1], reverse=True)[:top_k]

        return {
            'total_unique_patterns': len(meaningful_patterns),
            'top_ngrams': [{'pattern': pattern, 'count': count}
                          for pattern, count in top_patterns],
            'sample_size': len(sampled_logs)
        }

    def group_similar_logs(self, logs: List[Dict]) -> List[Dict]:
        """Group logs by security event type first"""
        if len(logs) > MAX_LOGS_FOR_ANALYSIS:
            logs = logs[:MAX_LOGS_FOR_ANALYSIS]

        type_groups = defaultdict(list)
        untyped_logs = []

        for log in logs:
            if log.get('event_type'):
                type_groups[log['event_type']].append(log)
            else:
                untyped_logs.append(log)

        result_groups = []

        for event_type, group_logs in type_groups.items():
            level_counts = defaultdict(int)
            for log in group_logs:
                level_counts[log['level']] += 1

            result_groups.append({
                'group_id': f"type_{event_type}",
                'event_type': event_type,
                'size': len(group_logs),
                'levels': dict(level_counts),
                'samples': group_logs[:3],
                'logs': group_logs
            })

        if untyped_logs:
            max_untyped = min(len(untyped_logs), 500)
            untyped_sample = untyped_logs[:max_untyped]

            # Group by level for untyped logs
            level_groups = defaultdict(list)
            for log in untyped_sample:
                level_groups[log['level']].append(log)

            for level, group_logs in level_groups.items():
                result_groups.append({
                    'group_id': f"level_{level}",
                    'event_type': 'UNKNOWN',
                    'size': len(group_logs),
                    'levels': {level: len(group_logs)},
                    'samples': group_logs[:3],
                    'logs': group_logs
                })

        result_groups.sort(key=lambda x: x['size'], reverse=True)

        return result_groups

    def analyze_group(self, group: Dict) -> str:
        """Fast LLM analysis with structured fallback"""

        event_type = group.get('event_type', 'UNKNOWN')
        size = group['size']
        levels = group.get('levels', {})

        # Build compact prompt
        prompt = f"""Event: {event_type}
Count: {size}
Levels: {levels}

Answer in 3 short lines:
1. Attack type
2. Severity
3. Action"""

        try:
            messages = [
                {"role": "system", "content": "Cybersecurity expert. Answer briefly."},
                {"role": "user", "content": prompt}
            ]

            formatted_prompt = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            inputs = self.tokenizer(formatted_prompt, return_tensors="pt", truncation=True, max_length=256)
            if DEVICE == "cuda":
                inputs = {k: v.cuda() for k, v in inputs.items()}

            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=120,
                    temperature=0.5,
                    do_sample=True,
                    top_p=0.9,
                    repetition_penalty=1.2,
                    pad_token_id=self.tokenizer.eos_token_id,
                    eos_token_id=self.tokenizer.eos_token_id
                )

            input_length = inputs['input_ids'].shape[1]
            response_tokens = outputs[0][input_length:]
            response = self.tokenizer.decode(response_tokens, skip_special_tokens=True)

            response = response.strip()

            if len(response) < 20:
                return self._generate_fallback_analysis(group)

            return response

        except Exception as e:
            print(f"LLM error: {e}")
            return self._generate_fallback_analysis(group)

    def _generate_fallback_analysis(self, group: Dict) -> str:
        """Structured fallback analysis"""
        event_type = group.get('event_type', 'UNKNOWN')
        size = group['size']
        levels = group.get('levels', {})

        severity = 'medium'
        if 'CRITICAL' in levels:
            severity = 'critical'
        elif 'ERROR' in levels:
            severity = 'high'
        elif 'WARNING' in levels:
            severity = 'medium'

        attack_info = {
            'DDOS': ('Distributed Denial of Service attack',
                     'Rate limiting, IP blocking, traffic filtering'),
            'XSS': ('Cross-Site Scripting attack',
                    'Input sanitization, CSP headers, WAF rules'),
            'SCALE_INJECTION': ('Resource scaling attack',
                                'Authentication, rate limits, resource quotas'),
            'TG': ('Threat group activity',
                   'Monitor traffic, update threat intelligence, block suspicious IPs'),
            'UNKNOWN': ('Unknown or unclassified event',
                        'Manual review required, check patterns')
        }

        if event_type in attack_info:
            description, actions = attack_info[event_type]
        else:
            description, actions = attack_info['UNKNOWN']

        return f"""1. Attack: {event_type} - {description}
2. Severity: {severity}
3. Events: {size} ({levels})
4. Actions: {actions}"""

In [ ]:
# @title 4. Analysis Function
def analyze_log_content(analyzer: LogAnalyzer, content: str) -> Dict:
    """Complete log analysis pipeline"""

    print("\n" + "="*60)
    print("LOG ANALYSIS")
    print("="*60)

    # Parse logs
    important_logs = analyzer.parse_logs(content)

    # Statistics
    level_stats = defaultdict(int)
    for log in important_logs:
        level_stats[log['level']] += 1

    print(f"\nTotal important events: {len(important_logs)}")
    print("Severity distribution:")

    severity_emoji = {'CRITICAL': '⛔', 'ERROR': '🔴', 'WARNING': '⚠️'}
    for level, count in sorted(level_stats.items(), key=lambda x: -x[1]):
        print(f"  {severity_emoji.get(level, '•')} {level}: {count}")

    if not important_logs:
        print("\nNo important events found!")
        return {}

    # Security event type statistics
    event_type_stats = defaultdict(int)
    for log in important_logs:
        event_type = log.get('event_type', 'UNKNOWN')
        event_type_stats[event_type] += 1

    print("\nSecurity Event Types:")
    total_important = len(important_logs)
    for event_type, count in sorted(event_type_stats.items(), key=lambda x: -x[1]):
        percentage = (count / total_important) * 100 if total_important > 0 else 0
        print(f"  🎯 {event_type}: {count} ({percentage:.1f}%)")

    # N-gram analysis
    print("\nAnalyzing common patterns (n-grams)...")
    ngram_analysis = analyzer.analyze_ngrams(important_logs, top_k=20)

    print("\nTop patterns:")
    for ngram in ngram_analysis['top_ngrams']:
        print(f"  • {ngram['pattern']}: {ngram['count']}")

    # Group similar logs
    print("\nGrouping logs...")
    groups = analyzer.group_similar_logs(important_logs)
    print(f"Found {len(groups)} groups")

    # Analyze top groups
    results = []
    max_groups = min(MAX_GROUPS_TO_ANALYZE, len(groups))

    print(f"\nAnalyzing top {max_groups} groups...")

    for i, group in enumerate(groups[:max_groups]):
        event_type = group.get('event_type', 'UNKNOWN')
        print(f"\n  Analyzing group {i+1}/{max_groups} ({event_type}, {group['size']} events)...")
        analysis = analyzer.analyze_group(group)

        results.append({
            'group_id': group['group_id'],
            'event_type': event_type,
            'group_size': group['size'],
            'levels': group['levels'],
            'samples': [log['message'][:200] for log in group['samples'][:2]],
            'analysis': analysis
        })

    return {
        'total_lines': len(content.strip().split('\n')),
        'total_important': len(important_logs),
        'level_stats': dict(level_stats),
        'event_type_stats': dict(event_type_stats),
        'ngram_analysis': ngram_analysis,
        'total_groups': len(groups),
        'analyzed_groups': results
    }

In [ ]:
# @title 5. Upload and Analyze Log Files (Folder Support)
from google.colab import files
import json
import os
from datetime import datetime

# Initialize analyzer
analyzer = LogAnalyzer()

print("\n" + "="*60)
print("LOG FOLDER UPLOAD")
print("="*60)
print("\nUpload .log files (can select multiple):")
print("Or upload a ZIP file with logs")
print("Supported: .log, .txt")

uploaded = files.upload()

all_results = []

for filename in uploaded.keys():
    print(f"\n{'='*60}")
    print(f"📄 File: {filename}")
    print(f"{'='*60}")

    if filename.endswith('.zip'):
        print("📦 ZIP file detected, extracting...")
        import zipfile
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('/content/logs')
        print("Extracted. Processing files...")

        for extracted_file in os.listdir('/content/logs'):
            if extracted_file.endswith(('.log', '.txt')):
                file_path = os.path.join('/content/logs', extracted_file)
                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    content = f.read()

                print(f"\n  Analyzing: {extracted_file}")
                results = analyze_log_content(analyzer, content)

                if results:
                    results['filename'] = extracted_file

                    level_stats = results.get('level_stats', {})
                    errors = level_stats.get('ERROR', 0)
                    criticals = level_stats.get('CRITICAL', 0)
                    warnings = level_stats.get('WARNING', 0)

                    results['has_errors'] = (errors + criticals) > 0
                    results['has_threats'] = (errors + criticals) > 0
                    results['error_count'] = errors + criticals
                    results['warning_count'] = warnings

                    all_results.append(results)
        continue

    if not filename.endswith(('.log', '.txt')):
        print(f"⚠️ Skipping non-log file: {filename}")
        continue

    try:
        with open(filename, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()

        line_count = len(content.strip().split('\n'))
        print(f"File size: {line_count} lines")

        results = analyze_log_content(analyzer, content)

        if results:
            results['filename'] = filename

            level_stats = results.get('level_stats', {})
            errors = level_stats.get('ERROR', 0)
            criticals = level_stats.get('CRITICAL', 0)
            warnings = level_stats.get('WARNING', 0)

            results['has_errors'] = (errors + criticals) > 0
            results['has_threats'] = (errors + criticals) > 0
            results['error_count'] = errors + criticals
            results['warning_count'] = warnings

            all_results.append(results)

    except Exception as e:
        print(f"Error processing file: {e}")

# Save combined results
if all_results:
    combined_output = {
        'analysis_time': datetime.now().isoformat(),
        'total_files': len(all_results),
        'files_with_errors': sum(1 for r in all_results if r['has_errors']),
        'files_with_warnings_only': sum(1 for r in all_results if not r['has_errors'] and r['warning_count'] > 0),
        'files_clean': sum(1 for r in all_results if not r['has_errors'] and r['warning_count'] == 0),
        'files': all_results
    }

    with open('combined_analysis.json', 'w', encoding='utf-8') as f:
        json.dump(combined_output, f, ensure_ascii=False, indent=2, default=str)

    print(f"\n{'='*60}")
    print(f"✅ ANALYSIS COMPLETE")
    print(f"{'='*60}")
    print(f"Total files: {len(all_results)}")
    print(f"Files with errors: {sum(1 for r in all_results if r['has_errors'])}")
    print(f"Files with warnings only: {sum(1 for r in all_results if not r['has_errors'] and r['warning_count'] > 0)}")
    print(f"Clean files: {sum(1 for r in all_results if not r['has_errors'] and r['warning_count'] == 0)}")

In [ ]:
# @title 6. Telegram Report Format
import json
import matplotlib.pyplot as plt
import textwrap

def format_telegram_message(results: Dict) -> str:
    """Format analysis results for Telegram"""

    if not results:
        return "❌ No results"

    # Determine status
    if results.get('has_threats'):
        status = "🔴 THREATS DETECTED"
    else:
        status = "✅ CLEAN"

    # Build message
    msg = f"""📋 File: {results.get('filename', 'Unknown')}
Status: {status}

📊 Statistics:
  • Lines: {results.get('total_lines', 0)}
  • Important events: {results.get('total_important', 0)}
  • Groups: {results.get('total_groups', 0)}

⚠️ Severity:
"""

    severity_emoji = {'CRITICAL': '⛔', 'ERROR': '🔴', 'WARNING': '⚠️'}
    level_stats = results.get('level_stats', {})
    for level, count in sorted(level_stats.items(), key=lambda x: -x[1]):
        msg += f"  {severity_emoji.get(level, '•')} {level}: {count}\n"

    # Event types
    event_type_stats = results.get('event_type_stats', {})
    event_type_stats = {k: v for k, v in event_type_stats.items() if k != 'UNKNOWN' and k is not None}

    if event_type_stats:
        msg += "\n🎯 Attacks:\n"
        for event_type, count in sorted(event_type_stats.items(), key=lambda x: -x[1]):
            msg += f"  • {event_type}: {count}\n"

    # Top analysis
    analyzed_groups = results.get('analyzed_groups', [])
    if analyzed_groups:
        msg += "\n💡 Top findings:\n"
        for group in analyzed_groups[:3]:
            event_type = group.get('event_type', 'UNKNOWN')
            size = group.get('group_size', 0)
            analysis = group.get('analysis', '')

            # Clean analysis
            analysis_clean = analysis[:100].replace('\n', ' ')

            msg += f"  • {event_type} ({size} events): {analysis_clean}...\n"

    return msg

def format_telegram_summary(all_results: List[Dict]) -> str:
    """Format summary for all files"""

    if not all_results:
        return "❌ No files analyzed"

    total_files = len(all_results)
    threat_files = [r for r in all_results if r.get('has_threats')]
    clean_files = [r for r in all_results if not r.get('has_threats')]

    msg = f"""🔒 LOG ANALYSIS SUMMARY
=======================

📁 Total files: {total_files}
🔴 With threats: {len(threat_files)}
✅ Clean: {len(clean_files)}

"""

    if threat_files:
        msg += "⚠️ FILES WITH THREATS:\n"
        for r in threat_files:
            filename = r.get('filename', 'Unknown')
            total_important = r.get('total_important', 0)
            msg += f"  🔴 {filename}: {total_important} events\n"

    if clean_files:
        msg += "\n✅ CLEAN FILES:\n"
        for r in clean_files:
            filename = r.get('filename', 'Unknown')
            msg += f"  • {filename}\n"

    return msg

# Display results
if 'all_results' in locals() and all_results:
    # Show summary first
    summary = format_telegram_summary(all_results)
    print(summary)

    print("\n" + "="*70)
    print("DETAILED FILE REPORTS")
    print("="*70)

    for result in all_results:
        print(f"\n{'─'*70}")
        telegram_msg = format_telegram_message(result)
        print(telegram_msg)
        print(f"{'─'*70}")

    # Save telegram-ready messages
    telegram_messages = []
    for result in all_results:
        telegram_messages.append(format_telegram_message(result))

    with open('telegram_messages.json', 'w', encoding='utf-8') as f:
        json.dump(telegram_messages, f, ensure_ascii=False, indent=2)

    print("\n✅ Telegram messages saved: telegram_messages.json")

elif 'results' in locals() and results:
    # Single file mode
    telegram_msg = format_telegram_message(results)
    print(telegram_msg)
else:
    print("Run the analysis first (previous cell)")

In [ ]:
# @title Send Results to Telegram with Level Buttons
import requests
import json
from datetime import datetime

# ========== SETTINGS ==========
BOT_TOKEN = "YOUR BOT TOKEN"
CHAT_ID = "YOUR CHAT ID" #check @GetMyID_bot
# ==============================

def send_telegram_message(bot_token: str, chat_id: str, text: str, reply_markup=None):
    """Send message to Telegram with optional buttons"""
    url = f"https://api.telegram.org/bot{bot_token}/sendMessage"

    max_length = 4000

    messages = []
    if len(text) > max_length:
        lines = text.split('\n')
        current = ""
        for line in lines:
            if len(current) + len(line) + 1 > max_length:
                messages.append(current)
                current = line
            else:
                current += line + "\n" if current else line
        if current:
            messages.append(current)
    else:
        messages = [text]

    results = []
    for i, msg in enumerate(messages):
        payload = {
            'chat_id': chat_id,
            'text': msg,
            'parse_mode': 'HTML',
            'disable_web_page_preview': True
        }

        if i == len(messages) - 1 and reply_markup:
            payload['reply_markup'] = json.dumps(reply_markup)

        response = requests.post(url, json=payload, timeout=10)

        if response.status_code == 200:
            print(f"✅ Message {i+1}/{len(messages)} sent")
            results.append(response.json())
        else:
            print(f"❌ Failed: {response.text}")
            results.append(None)

    return results

def create_level_buttons(filename: str, level_stats: Dict) -> Dict:
    """Create buttons for each severity level"""
    buttons = []

    level_emoji = {'CRITICAL': '⛔', 'ERROR': '🔴', 'WARNING': '⚠️'}

    for level, count in sorted(level_stats.items(), key=lambda x: -x[1]):
        if count > 0:
            buttons.append([
                {
                    'text': f"{level_emoji.get(level, '•')} {level}: {count} events",
                    'callback_data': f"explain:{filename}:{level}"
                }
            ])

    return {'inline_keyboard': buttons}

def format_file_report_telegram(results: Dict) -> str:
    """Format single file report for Telegram"""
    if not results:
        return "❌ No results"

    filename = results.get('filename', 'Unknown')
    total_lines = results.get('total_lines', 0)
    error_count = results.get('error_count', 0)
    warning_count = results.get('warning_count', 0)

    if error_count > 0:
        status = "🔴 ERRORS DETECTED"
    elif warning_count > 0:
        status = "⚠️ WARNINGS ONLY"
    else:
        status = "✅ CLEAN"

    msg = f"""📄 <b>{filename}</b>
{status}

📊 Statistics:
  • Lines: {total_lines}
  • Errors (ERROR+CRITICAL): {error_count}
  • Warnings: {warning_count}
"""

    level_stats = results.get('level_stats', {})
    if level_stats:
        msg += "\n⚠️ <b>Levels:</b>\n"
        severity_emoji = {'CRITICAL': '⛔', 'ERROR': '🔴', 'WARNING': '⚠️'}
        for level, count in sorted(level_stats.items(), key=lambda x: -x[1]):
            msg += f"  {severity_emoji.get(level, '•')} {level}: {count}\n"

    return msg

def format_summary_telegram(all_results: List) -> str:
    """Format summary for all files"""
    if not all_results:
        return "❌ No files analyzed"

    total_files = len(all_results)
    error_files = [r for r in all_results if r.get('has_errors', False)]
    warning_only_files = [r for r in all_results if not r.get('has_errors', False) and r.get('warning_count', 0) > 0]
    clean_files = [r for r in all_results if not r.get('has_errors', False) and r.get('warning_count', 0) == 0]

    msg = f"""🔒 <b>LOG ANALYSIS REPORT</b>
🕐 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

📁 Total files: {total_files}
🔴 With errors: {len(error_files)}
⚠️ Warnings only: {len(warning_only_files)}
✅ Clean: {len(clean_files)}

"""

    if error_files:
        msg += "🔴 <b>FILES WITH ERRORS:</b>\n"
        for r in error_files:
            filename = r.get('filename', 'Unknown')
            errors = r.get('error_count', 0)
            msg += f"  🔴 {filename}: {errors} errors\n"

    if warning_only_files:
        msg += "\n⚠️ <b>FILES WITH WARNINGS:</b>\n"
        for r in warning_only_files:
            filename = r.get('filename', 'Unknown')
            warnings = r.get('warning_count', 0)
            msg += f"  ⚠️ {filename}: {warnings} warnings\n"

    if clean_files:
        msg += "\n✅ <b>CLEAN FILES:</b>\n"
        for r in clean_files:
            filename = r.get('filename', 'Unknown')
            msg += f"  • {filename}\n"

    return msg

# ========== SEND ==========

print("📤 Sending results to Telegram...")

if 'all_results' in locals() and all_results:
    # Send summary
    summary = format_summary_telegram(all_results)
    print("\nSending summary...")
    send_telegram_message(BOT_TOKEN, CHAT_ID, summary)

    # Send detailed report for files with errors
    error_files = [r for r in all_results if r.get('has_errors', False)]

    for result in error_files:
        print(f"\nSending report for: {result.get('filename')}")
        file_report = format_file_report_telegram(result)

        # Create buttons for each level
        level_stats = result.get('level_stats', {})
        buttons = create_level_buttons(result.get('filename', 'unknown'), level_stats)

        send_telegram_message(BOT_TOKEN, CHAT_ID, file_report, buttons)

    print("\n✅ All messages sent to Telegram!")

else:
    print("❌ No results to send. Run analysis first.")

In [ ]:
# @title Telegram Bot Callback Handler (WORKING VERSION)
import requests
import json
import time

# ========== SETTINGS ==========
BOT_TOKEN = "YOUR BOT TOKEN"
# ==============================

# Store explanations globally
explanations_cache = {}

def get_updates(offset=None):
    """Get bot updates"""
    url = f"https://api.telegram.org/bot{BOT_TOKEN}/getUpdates"
    params = {
        'timeout': 30,
        'allowed_updates': json.dumps(['callback_query', 'message'])
    }
    if offset:
        params['offset'] = offset

    response = requests.get(url, params=params, timeout=35)
    return response.json()

def answer_callback(callback_id: str, text: str = ""):
    """Answer callback query (required to remove loading state)"""
    url = f"https://api.telegram.org/bot{BOT_TOKEN}/answerCallbackQuery"
    payload = {
        'callback_query_id': callback_id,
        'text': text
    }
    requests.post(url, json=payload, timeout=5)

def send_message(chat_id: str, text: str):
    """Send message to chat"""
    url = f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage"
    payload = {
        'chat_id': chat_id,
        'text': text,
        'parse_mode': 'HTML',
        'disable_web_page_preview': True
    }
    return requests.post(url, json=payload, timeout=10)

def get_level_explanation(filename: str, level: str) -> str:
    """Get explanation for specific severity level"""

    # Check cache first
    cache_key = f"{filename}_{level}"
    if cache_key in explanations_cache:
        return explanations_cache[cache_key]

    # Search in all_results (global)
    all_results = globals().get('all_results', [])

    for result in all_results:
        if result.get('filename') == filename:
            level_stats = result.get('level_stats', {})
            count = level_stats.get(level, 0)

            level_emoji = {'CRITICAL': '⛔', 'ERROR': '🔴', 'WARNING': '⚠️'}
            emoji = level_emoji.get(level, '•')

            # Get sample logs for this level
            samples = []
            for group in result.get('analyzed_groups', []):
                group_levels = group.get('levels', {})
                if group_levels.get(level, 0) > 0:
                    for sample in group.get('samples', [])[:3]:
                        samples.append(sample[:120])
                    break

            samples_text = "\n".join([f"  • {s}" for s in samples[:3]]) if samples else "  No samples available"

            # Get analysis
            analysis_text = "No analysis available"
            for group in result.get('analyzed_groups', []):
                group_levels = group.get('levels', {})
                if group_levels.get(level, 0) > 0:
                    analysis_text = group.get('analysis', 'No analysis')
                    break

            msg = f"""{emoji} <b>{level} Events in {filename}</b>

📊 <b>Count:</b> {count}

📝 <b>Sample logs:</b>
{samples_text}

💡 <b>Analysis:</b>
{analysis_text[:300]}

🔧 <b>What to do:</b>"""

            if level == 'CRITICAL':
                msg += "\n1. Check system immediately\n2. Review error logs\n3. Contact admin"
            elif level == 'ERROR':
                msg += "\n1. Investigate errors\n2. Check affected services\n3. Fix root cause"
            elif level == 'WARNING':
                msg += "\n1. Monitor situation\n2. Check patterns\n3. No immediate action needed"

            # Cache the result
            explanations_cache[cache_key] = msg

            return msg

    return "❌ Data not found. Run analysis first."

# Main polling loop
print("🔄 Bot is listening for button clicks...")
print("Press Stop button in Colab to stop")

last_update_id = None

while True:
    try:
        updates = get_updates(offset=last_update_id)

        if updates.get('ok') and updates.get('result'):
            for update in updates['result']:
                update_id = update['update_id']
                last_update_id = update_id + 1

                # Handle callback query
                if 'callback_query' in update:
                    callback = update['callback_query']
                    callback_id = callback['id']
                    callback_data = callback.get('data', '')
                    chat_id = callback.get('message', {}).get('chat', {}).get('id')

                    print(f"🔘 Button clicked: {callback_data}")

                    # Parse: explain:filename:level
                    parts = callback_data.split(':')

                    if len(parts) == 3 and parts[0] == 'explain':
                        filename = parts[1]
                        level = parts[2]

                        # Answer callback immediately (removes loading)
                        answer_callback(callback_id, f"Generating {level} explanation...")

                        # Get explanation
                        explanation = get_level_explanation(filename, level)

                        # Send explanation
                        send_message(chat_id, explanation)

                        print(f"✅ Explanation sent for {filename} - {level}")
                    else:
                        answer_callback(callback_id, "Unknown action")

        time.sleep(1)

    except Exception as e:
        print(f"Error: {e}")
        time.sleep(3)